# 04 - Match AFC buses to AVL vehicles and materialize trip positions

Bridges `ml.trip_validity_trips.bus_id` (AFC's `vehicle_number`, 5-digit
zero-padded) to `silver.avl_pings`' own vehicle identity, then
materializes every trip's GPS trace for fast per-trip lookup later
(map rendering). Two new tables, both read by `05_final_dataset.ipynb`:

- `ml.trip_validity_bus_avl_match`: one row per distinct `bus_id`, the
  resolved AVL identity (if any) and which crosswalk resolved it.
- `ml.trip_validity_trip_positions`: one row per AVL ping belonging to
  a matched trip, within that trip's own `[trip_opening_timestamp,
  trip_closing_timestamp]` window - no padding.

## The two crosswalks

`silver.avl_pings` identifies a vehicle by `vehicle_id` (integer) *or*
`device_id` (text, e.g. `"ep1-428113843"`, the GPS hardware unit's own
id). Two silver reference tables bridge AFC's `bus_id` to one or the
other, confirmed live against the DB (this notebook is the only one
touching `silver.dictionary_vehicle`/`silver.dictionary_device`/
`silver.avl_pings`):

- `silver.dictionary_vehicle` (4,530 rows, 1 snapshot): `cod_veiculo`
  is free-text and needs stripping to digits only (`"02018v"` ->
  `"02018"`, `"02008 - Desativado"` -> `"02008"`; 2,122/4,530 rows carry
  this kind of noise) before it lines up with `bus_id`'s own padding.
  `id_veiculo` is always plain-integer text and equals
  `avl_pings.vehicle_id` once cast.
- `silver.dictionary_device` (2,219 rows, 1 snapshot): `codigo` is kept
  only where it's a plain integer already (`codigo ~ '^[0-9]+$'`,
  2,177/2,219 rows qualify - the rest are dropped, not cleaned, since
  they're not vehicle numbers at all) and `device_id` is non-null.
  `device_id` matches `avl_pings.device_id` directly, same format,
  confirmed by spot-check - no transform needed.

Neither crosswalk guarantees a unique `bus_id`: `dictionary_vehicle`
has known bus-reassignment noise (documented on
`contracts/vehicle_dictionary.py`), and a few `dictionary_device` rows
share a `codigo`. Per instruction, duplicates are resolved by picking
one arbitrarily but deterministically (`DISTINCT ON`), not by trying to
decide which is "correct" - that's out of scope here.

**Priority when both crosswalks resolve the same `bus_id`**:
`dictionary_device` wins (its `device_id` is used); `dictionary_vehicle`
is only used as a fallback when no `dictionary_device` match exists.

## Why this needs new, covering indexes first

`silver.avl_pings` is **1.6 billion rows** across all of 2023;
November 2023 alone (the only period `ml.trip_validity_trips` covers)
is a single 21 GB partition (`avl_pings_y2023m11`) with
**130,957,355 rows**, both confirmed live. Today it's indexed only on
`geom` (GiST) and `metric_timestamp` (btree) - nothing on `vehicle_id`
or `device_id`, so matching ~941K trips by vehicle+time would mean
scanning the whole partition. `src/opa_database/silver/avl.py` now
defines two composite indexes, `(vehicle_id, metric_timestamp)` and
`(device_id, metric_timestamp)`, added to its `_INDEXES` tuple so every
*future* monthly AVL load gets them automatically - but that doesn't
retroactively touch the already-loaded November 2023 partition, so
Stage 0 below backfills the same two indexes onto it directly, once,
using the exact index names/definitions `IndexSpec` would generate
(`{partition}_{suffix}`) so a future reload of this same partition
would recreate indistinguishable indexes rather than duplicates.

Both indexes also carry `latitude`/`longitude`/`speed`/`odometer` as
`INCLUDE` columns (confirmed live via `EXPLAIN (ANALYZE, BUFFERS)`
during development): without them, Postgres still has to fetch the
heap page for every matching row just to read those four columns, and
since pings for the same vehicle aren't stored contiguously (insert
order is global ingestion order, not per-vehicle), that's effectively
random I/O across a 21 GB table - the dominant real cost of Stage 2's
insert. With them, the whole query is answerable from the index alone
(an index-only scan), skipping the heap entirely.

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg import sql

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
print("connected")

connected


## Stage 0 - backfill the covering indexes onto `avl_pings_y2023m11`

`CREATE INDEX CONCURRENTLY` avoids locking the partition against other
users of this shared database, but can't run inside a transaction block
- this needs its own `autocommit=True` connection, separate from `conn`
used everywhere else in this notebook. `IF NOT EXISTS` makes this safe
to re-run. Expect real wall-clock time here: two btree indexes over
130,957,355 rows, wider than a plain composite index because of the
`INCLUDE` columns.

In [4]:
import time

autocommit_conn = psycopg.connect(settings.db_dsn, autocommit=True)

_INCLUDE_COLS = "INCLUDE (latitude, longitude, speed, odometer)"

for index_name, definition in (
    (
        "avl_pings_y2023m11_vehicle_ts_idx",
        f"(vehicle_id, metric_timestamp) {_INCLUDE_COLS}",
    ),
    (
        "avl_pings_y2023m11_device_ts_idx",
        f"(device_id, metric_timestamp) {_INCLUDE_COLS}",
    ),
):
    start = time.monotonic()
    autocommit_conn.execute(
        sql.SQL(
            "CREATE INDEX CONCURRENTLY IF NOT EXISTS {name} "
            "ON silver.avl_pings_y2023m11 {definition}"
        ).format(name=sql.Identifier(index_name), definition=sql.SQL(definition))
    )
    print(f"{index_name}: {time.monotonic() - start:.1f}s")

autocommit_conn.close()
print("index backfill done")

avl_pings_y2023m11_vehicle_ts_idx: 0.0s
avl_pings_y2023m11_device_ts_idx: 0.0s
index backfill done


## Stage 1 - `ml.trip_validity_bus_avl_match`

One row per distinct `bus_id` actually present in
`ml.trip_validity_trips` (thousands of rows, not hundreds of millions -
cheap). Two candidate CTEs, each deduplicated to one row per normalized
bus_id via `DISTINCT ON` (arbitrary but deterministic tie-break, per
instruction - it doesn't matter which duplicate wins), then combined
with `dictionary_device` preferred over `dictionary_vehicle`.

In [5]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_bus_avl_match CASCADE;

    CREATE TABLE ml.trip_validity_bus_avl_match (
        bus_id            text PRIMARY KEY,
        avl_matched       boolean NOT NULL,
        avl_match_source  text CHECK (
                              avl_match_source IN
                              ('dictionary_vehicle', 'dictionary_device')
                          ),
        avl_vehicle_id    integer,
        avl_device_id     text
    );
""")

conn.execute("""
    WITH vehicle_candidates AS (
        SELECT DISTINCT ON (bus_id) bus_id, id_veiculo::integer AS avl_vehicle_id
        FROM (
            SELECT
                CASE WHEN length(norm) < 5 THEN lpad(norm, 5, '0') ELSE norm END
                    AS bus_id,
                id_veiculo
            FROM (
                SELECT regexp_replace(cod_veiculo, '[^0-9]', '', 'g') AS norm,
                       id_veiculo
                FROM silver.dictionary_vehicle
            ) normalized
            WHERE norm <> ''
        ) padded
        ORDER BY bus_id, id_veiculo
    ),
    device_candidates AS (
        SELECT DISTINCT ON (bus_id) bus_id, device_id AS avl_device_id
        FROM (
            SELECT
                CASE WHEN length(codigo) < 5 THEN lpad(codigo, 5, '0') ELSE codigo END
                    AS bus_id,
                device_id
            FROM silver.dictionary_device
            WHERE codigo ~ '^[0-9]+$' AND device_id IS NOT NULL
        ) padded
        ORDER BY bus_id, device_id
    ),
    distinct_bus_ids AS (
        SELECT DISTINCT bus_id FROM ml.trip_validity_trips
    )
    INSERT INTO ml.trip_validity_bus_avl_match (
        bus_id, avl_matched, avl_match_source, avl_vehicle_id, avl_device_id
    )
    SELECT
        b.bus_id,
        (dc.avl_device_id IS NOT NULL OR vc.avl_vehicle_id IS NOT NULL) AS avl_matched,
        CASE
            WHEN dc.avl_device_id IS NOT NULL THEN 'dictionary_device'
            WHEN vc.avl_vehicle_id IS NOT NULL THEN 'dictionary_vehicle'
        END AS avl_match_source,
        CASE WHEN dc.avl_device_id IS NULL THEN vc.avl_vehicle_id END AS avl_vehicle_id,
        dc.avl_device_id
    FROM distinct_bus_ids b
    LEFT JOIN device_candidates dc ON dc.bus_id = b.bus_id
    LEFT JOIN vehicle_candidates vc ON vc.bus_id = b.bus_id;
""")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            count(*) AS total,
            count(*) FILTER (
                WHERE avl_match_source = 'dictionary_device'
            ) AS via_device,
            count(*) FILTER (
                WHERE avl_match_source = 'dictionary_vehicle'
            ) AS via_vehicle,
            count(*) FILTER (WHERE NOT avl_matched) AS unmatched
        FROM ml.trip_validity_bus_avl_match;
    """)
    print("total / via_device / via_vehicle / unmatched:", cur.fetchone())

total / via_device / via_vehicle / unmatched: (1730, 1461, 204, 65)


## Stage 2 - `ml.trip_validity_trip_positions`

One row per AVL ping within a matched trip's exact
`[trip_opening_timestamp, trip_closing_timestamp]` window.
`PRIMARY KEY (trip_id, metric_timestamp)` alone gives fast,
naturally time-ordered per-trip retrieval - the exact "join from the
dataset to get this trip's locations" access pattern - so no separate
`trip_id`-only index is needed. A GiST index on `geom` is added too,
matching this project's convention for point/line geometry columns, for
any future spatial (not just per-trip) queries.

The 9 trips with the `1899-12-30` Delphi zero-date sentinel as
`trip_closing_timestamp` naturally produce zero rows here (`closing <
opening` is an empty window) - no special-casing needed.

`silver.avl_pings` itself isn't unique on `(vehicle_id/device_id,
metric_timestamp)` - confirmed live, e.g. two literally-identical rows
for device `ep1-428115569` at `2023-11-22 22:18:23+00` - so the insert
below uses `ON CONFLICT (trip_id, metric_timestamp) DO NOTHING` to keep
just one arbitrarily (doesn't matter which, since the DB doesn't say
which copy is "real"). This is cheaper than an explicit
`DISTINCT ON`/dedup pass, which would force sorting the entire result
set before insert even though real duplicates are rare - `ON CONFLICT`
only pays a cost on an actual collision, checked via the primary key
index incrementally as rows stream in.

In [6]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_trip_positions CASCADE;

    CREATE TABLE ml.trip_validity_trip_positions (
        trip_id           bigint NOT NULL
                              REFERENCES ml.trip_validity_trips (trip_id),
        metric_timestamp  timestamptz NOT NULL,
        latitude          double precision NOT NULL,
        longitude         double precision NOT NULL,
        speed             integer NOT NULL,
        odometer          bigint NOT NULL,
        geom              geometry(Point, 4326) GENERATED ALWAYS AS (
                              ST_SetSRID(ST_MakePoint(longitude, latitude), 4326)
                          ) STORED,
        PRIMARY KEY (trip_id, metric_timestamp)
    );
""")
conn.commit()
print("ml.trip_validity_trip_positions created")

ml.trip_validity_trip_positions created


### Timing sample before the full run

Same honesty convention as `03_trip_metrics.ipynb`'s heaviest stage:
time the exact join pattern against a random 2,000-trip sample first via
a temp table, then extrapolate to all matched trips, rather than just
starting the full run and hoping.

**Two planner traps found the hard way while building this notebook,
both confirmed live via `EXPLAIN (ANALYZE, BUFFERS)`** - documented here
since the fix looks unusual without the reasoning:

1. A single query with `... AND (device_id = X OR vehicle_id = Y) ...`
   (mirroring `avl_match_source` picking one branch per row) makes
   Postgres unable to generate a parameterized index path for either
   branch once the comparison values come from a real outer table
   (not a literal/`VALUES` list) - it falls back to sequentially
   scanning the entire multi-hundred-million-row `avl_pings` as the
   *outer* side of the join and materializing the small trip sample as
   the *inner* side, once per `avl_pings` row. Fixed by splitting into
   a `UNION ALL` of two branches, each with exactly one equality
   condition (`device_id = X` in one, `vehicle_id = Y` in the other) -
   `avl_match_source` guarantees they're mutually exclusive, so
   `UNION ALL` never double-counts.
2. Even with the OR gone, Postgres still chooses a `Hash Join` over
   the *entire* partition instead of a nested loop with an indexed,
   per-row lookup - because it can only estimate `device_id`/
   `vehicle_id` equality selectivity on its own (poor: each vehicle
   has ~59K pings across all of November), not combined with a
   per-row time window it has no histogram for (each trip's real
   window is tiny). `SET LOCAL enable_hashjoin = off` /
   `enable_mergejoin = off` (scoped to just this transaction, reverted
   automatically on commit) forces the only remaining option: a nested
   loop using the new composite indexes per row. Confirmed via
   `EXPLAIN ANALYZE` on a 200-trip sample: 214ms total (most of it
   one-time JIT compilation), vs. 23+ minutes still running when this
   was first tried without either fix - long enough that it had to be
   cancelled via `pg_cancel_backend` rather than left to finish.

In [7]:
SAMPLE_SIZE = 2000

with conn.cursor() as cur:
    cur.execute(
        """
        CREATE TEMP TABLE avl_timing_sample AS
        SELECT t.trip_id, t.trip_opening_timestamp, t.trip_closing_timestamp,
               m.avl_match_source, m.avl_vehicle_id, m.avl_device_id
        FROM ml.trip_validity_trips t
        JOIN ml.trip_validity_bus_avl_match m
          ON m.bus_id = t.bus_id AND m.avl_matched
        ORDER BY random()
        LIMIT %(sample_size)s;
    """,
        {"sample_size": SAMPLE_SIZE},
    )
    cur.execute("ANALYZE avl_timing_sample;")
    cur.execute("SET LOCAL enable_hashjoin = off;")
    cur.execute("SET LOCAL enable_mergejoin = off;")

    start = time.monotonic()
    cur.execute("""
        SELECT count(*) FROM (
            SELECT p.metric_timestamp
            FROM avl_timing_sample s
            JOIN silver.avl_pings p
              ON p.device_id = s.avl_device_id
             AND p.metric_timestamp >= s.trip_opening_timestamp
             AND p.metric_timestamp <= s.trip_closing_timestamp
             AND p.metric_timestamp >= '2023-11-01'
             AND p.metric_timestamp < '2023-12-02'
            WHERE s.avl_match_source = 'dictionary_device'
            UNION ALL
            SELECT p.metric_timestamp
            FROM avl_timing_sample s
            JOIN silver.avl_pings p
              ON p.vehicle_id = s.avl_vehicle_id
             AND p.metric_timestamp >= s.trip_opening_timestamp
             AND p.metric_timestamp <= s.trip_closing_timestamp
             AND p.metric_timestamp >= '2023-11-01'
             AND p.metric_timestamp < '2023-12-02'
            WHERE s.avl_match_source = 'dictionary_vehicle'
        ) x;
    """)
    sample_positions = cur.fetchone()[0]
    sample_elapsed = time.monotonic() - start

    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_trips t
        JOIN ml.trip_validity_bus_avl_match m
          ON m.bus_id = t.bus_id AND m.avl_matched;
    """)
    total_matched_trips = cur.fetchone()[0]

    cur.execute("DROP TABLE avl_timing_sample;")
conn.commit()

per_trip_seconds = sample_elapsed / SAMPLE_SIZE
estimated_total_seconds = per_trip_seconds * total_matched_trips
print(f"{SAMPLE_SIZE} trips -> {sample_positions} positions in {sample_elapsed:.2f}s")
print(f"total matched trips: {total_matched_trips}")
print(f"estimated full run: {estimated_total_seconds / 60:.1f} minutes")

2000 trips -> 172031 positions in 0.15s
total matched trips: 931005
estimated full run: 1.2 minutes


### Full run

Runs in batches over the matched trips (not one giant statement), each
batch a small `INSERT ... SELECT ... UNION ALL ...` (one branch per
match source, `ON CONFLICT DO NOTHING` for the rare duplicate ping)
with `SET LOCAL enable_hashjoin = off` / `enable_mergejoin = off`
scoped to just that batch's transaction - see the timing-sample cell
above for why both are needed to get Postgres to actually use the
covering indexes via a nested loop, instead of sequentially scanning
the whole partition. The static `metric_timestamp >= '2023-11-01' AND <
'2023-12-02'` bound (redundant with the per-row window, which is always
inside November 2023) guarantees partition pruning down to just
`avl_pings_y2023m11` at plan time.

Batching exists for progress visibility, not correctness: a single
`INSERT` gives no way to check progress mid-run short of guessing from
`EXPLAIN`'s row estimates, which was a real problem earlier - a run
just looks identically "still going" whether it's 10% done or 95% done.
Each batch's query is tagged with a SQL comment (`/* batch i/N */`),
so progress is checkable anytime from *outside* this notebook via
`SELECT query FROM pg_stat_activity WHERE ...` - no separate log file
or table needed, and it's the same live-inspection approach already
used throughout this notebook's development.

**A third planner trap, found while testing this exact batched form**:
the obvious way to pass a batch is `WHERE t.trip_id = ANY(%(batch)s)`.
That alone silently degrades - psycopg infers the *narrowest* array
type that fits the values (`smallint[]`, since trip_ids fit), compared
against `trip_id`'s real type `bigint`, pushing the check from an
`Index Cond` to an un-indexed `Filter`. Casting explicitly
(`ANY(%(batch)s::bigint[])`) fixes *that* mismatch but uncovers a
second, deeper issue: even with the right type, Postgres treats a bound
array parameter as opaque at plan time and can choose to sequentially
scan all ~941K trips per batch rather than use the primary key index -
confirmed live via `EXPLAIN (ANALYZE, BUFFERS)`, a 6.5-second scan for
one branch of one 5,000-row batch. Batch 1 stalled for 9+ minutes on
the original form with no error, only found by comparing the *actual*
psycopg-parameterized query's plan (via a throwaway script) against the
literal-array version that worked fine.

The fix that actually works: build each batch as an inline `VALUES`
CTE (`WITH b (...) AS (VALUES (...), (...), ...)`), with the batch's
own rows composed as literals via `sql.Literal` (safe against
injection - values are DB data, not attacker input, but composed
properly regardless). A `VALUES` list is real, visible data to the
planner, not an opaque parameter, so it gets treated like the earlier
temp-table tests that worked - confirmed via the same script: the exact
5,000-row batch that took 6.5s+ per branch with `ANY()` completed in
0.14s total as a `VALUES` CTE.

In [8]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT t.trip_id, t.trip_opening_timestamp, t.trip_closing_timestamp,
               m.avl_match_source, m.avl_vehicle_id, m.avl_device_id
        FROM ml.trip_validity_trips t
        JOIN ml.trip_validity_bus_avl_match m
          ON m.bus_id = t.bus_id AND m.avl_matched
        ORDER BY t.trip_id;
    """)
    matched_trips = cur.fetchall()

BATCH_SIZE = 5000
batches = [
    matched_trips[i : i + BATCH_SIZE] for i in range(0, len(matched_trips), BATCH_SIZE)
]
print(
    f"{len(matched_trips)} matched trips, {len(batches)} batches of up to {BATCH_SIZE}"
)

931005 matched trips, 187 batches of up to 5000


In [9]:
def build_batch_insert(
    batch: list[tuple], batch_num: int, n_batches: int
) -> sql.Composed:
    """Build one batch's INSERT as a VALUES CTE, not an ANY() array parameter.

    psycopg's array-parameter binding turned out to be a planner trap in
    its own right (see markdown above) - a VALUES CTE with the batch's
    rows composed as literals keeps the data visible to the planner at
    plan time, which is what actually gets the covering-index nested
    loop chosen.
    """
    values_rows = sql.SQL(", ").join(
        sql.SQL("({}, {}, {}, {}, {}, {})").format(
            sql.Literal(trip_id),
            sql.Literal(opened),
            sql.Literal(closed),
            sql.Literal(match_source),
            sql.Literal(vehicle_id),
            sql.Literal(device_id),
        )
        for trip_id, opened, closed, match_source, vehicle_id, device_id in batch
    )
    comment = sql.SQL(f"/* trip_positions batch {batch_num}/{n_batches} */\n")
    body = sql.SQL("""
        WITH b (trip_id, trip_opening_timestamp, trip_closing_timestamp,
                avl_match_source, avl_vehicle_id, avl_device_id) AS (
            VALUES {values}
        )
        INSERT INTO ml.trip_validity_trip_positions (
            trip_id, metric_timestamp, latitude, longitude, speed, odometer
        )
        SELECT b.trip_id, p.metric_timestamp, p.latitude, p.longitude,
               p.speed, p.odometer
        FROM b
        JOIN silver.avl_pings p
          ON p.device_id = b.avl_device_id
         AND p.metric_timestamp >= b.trip_opening_timestamp
         AND p.metric_timestamp <= b.trip_closing_timestamp
         AND p.metric_timestamp >= '2023-11-01'
         AND p.metric_timestamp < '2023-12-02'
        WHERE b.avl_match_source = 'dictionary_device'

        UNION ALL

        SELECT b.trip_id, p.metric_timestamp, p.latitude, p.longitude,
               p.speed, p.odometer
        FROM b
        JOIN silver.avl_pings p
          ON p.vehicle_id = b.avl_vehicle_id
         AND p.metric_timestamp >= b.trip_opening_timestamp
         AND p.metric_timestamp <= b.trip_closing_timestamp
         AND p.metric_timestamp >= '2023-11-01'
         AND p.metric_timestamp < '2023-12-02'
        WHERE b.avl_match_source = 'dictionary_vehicle'

        ON CONFLICT (trip_id, metric_timestamp) DO NOTHING;
    """).format(values=values_rows)
    return comment + body


overall_start = time.monotonic()
for i, batch in enumerate(batches, start=1):
    batch_start = time.monotonic()
    with conn.cursor() as cur:
        cur.execute("SET LOCAL enable_hashjoin = off;")
        cur.execute("SET LOCAL enable_mergejoin = off;")
        cur.execute("SET LOCAL jit = off;")
        cur.execute(build_batch_insert(batch, i, len(batches)))
    conn.commit()

    elapsed_total = time.monotonic() - overall_start
    avg_per_batch = elapsed_total / i
    eta_minutes = avg_per_batch * (len(batches) - i) / 60
    print(
        f"batch {i}/{len(batches)}: {time.monotonic() - batch_start:.1f}s "
        f"(total {elapsed_total / 60:.1f}min, ETA {eta_minutes:.1f}min)",
        flush=True,
    )

print(
    f"positions inserted in {(time.monotonic() - overall_start) / 60:.1f} minutes total"
)

conn.execute("""
    CREATE INDEX trip_validity_trip_positions_geom_idx
        ON ml.trip_validity_trip_positions USING GIST (geom);
    ANALYZE ml.trip_validity_trip_positions;
""")
conn.commit()
print("index created, table analyzed")

batch 1/187: 2.3s (total 0.0min, ETA 7.2min)


batch 2/187: 2.3s (total 0.1min, ETA 7.1min)


batch 3/187: 2.3s (total 0.1min, ETA 7.0min)


batch 4/187: 2.3s (total 0.2min, ETA 7.0min)


batch 5/187: 2.3s (total 0.2min, ETA 7.0min)


...

batch 185/187: 2.3s (total 6.8min, ETA 0.1min)


batch 186/187: 2.3s (total 6.8min, ETA 0.0min)


batch 187/187: 0.4s (total 6.8min, ETA 0.0min)


positions inserted in 6.8 minutes total


index created, table analyzed


## Column-level provenance comments

In [10]:
def comment_on_column(cur: psycopg.Cursor, table: str, col: str, text: str) -> None:
    """Apply a COMMENT ON COLUMN for one column via safe SQL composition."""
    cur.execute(
        sql.SQL("COMMENT ON COLUMN ml.{}.{} IS {};").format(
            sql.Identifier(table), sql.Identifier(col), sql.Literal(text)
        )
    )


MATCH_COMMENTS = {
    "bus_id": "= ml.trip_validity_trips.bus_id. PK here too.",
    "avl_matched": (
        "True iff this bus_id resolved to an AVL vehicle_id or device_id "
        "via either crosswalk. False (never NULL) otherwise."
    ),
    "avl_match_source": (
        "Which silver crosswalk resolved the match: 'dictionary_device' "
        "(preferred when both match) or 'dictionary_vehicle' (fallback, "
        "used only when no dictionary_device match exists). NULL iff "
        "avl_matched is false."
    ),
    "avl_vehicle_id": (
        "silver.avl_pings.vehicle_id to join positions on, from "
        "silver.dictionary_vehicle.id_veiculo. Populated only when "
        "avl_match_source = 'dictionary_vehicle'."
    ),
    "avl_device_id": (
        "silver.avl_pings.device_id to join positions on, from "
        "silver.dictionary_device.device_id. Populated only when "
        "avl_match_source = 'dictionary_device'."
    ),
}
POSITIONS_COMMENTS = {
    "trip_id": "= ml.trip_validity_trips.trip_id (FK). Part of the PK.",
    "metric_timestamp": (
        "silver.avl_pings.metric_timestamp (UTC), as-is. Part of the PK "
        "- PRIMARY KEY (trip_id, metric_timestamp) makes per-trip "
        "retrieval a single ordered index range scan."
    ),
    "latitude": "silver.avl_pings.latitude, as-is.",
    "longitude": "silver.avl_pings.longitude, as-is.",
    "speed": "silver.avl_pings.speed, as-is.",
    "odometer": "silver.avl_pings.odometer, as-is.",
    "geom": (
        "GENERATED ALWAYS AS ST_SetSRID(ST_MakePoint(longitude, "
        "latitude), 4326) STORED, same convention as silver.avl_pings' "
        "own geom column."
    ),
}

with conn.cursor() as cur:
    for col, text in MATCH_COMMENTS.items():
        comment_on_column(cur, "trip_validity_bus_avl_match", col, text)
    for col, text in POSITIONS_COMMENTS.items():
        comment_on_column(cur, "trip_validity_trip_positions", col, text)

conn.execute(
    sql.SQL("COMMENT ON TABLE ml.trip_validity_bus_avl_match IS {};").format(
        sql.Literal(
            "One row per distinct bus_id in ml.trip_validity_trips, "
            "resolved (if possible) to an AVL vehicle_id or device_id via "
            "silver.dictionary_device (preferred) or "
            "silver.dictionary_vehicle (fallback). See "
            "ml/trip_validity_model/notebooks/04_avl_positions.ipynb."
        )
    )
)
conn.execute(
    sql.SQL("COMMENT ON TABLE ml.trip_validity_trip_positions IS {};").format(
        sql.Literal(
            "One row per AVL ping within a matched trip's exact "
            "[trip_opening_timestamp, trip_closing_timestamp] window, no "
            "padding. See ml/trip_validity_model/notebooks/"
            "04_avl_positions.ipynb."
        )
    )
)
conn.commit()
print("comments applied")

comments applied


## Verification

In [11]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT count(DISTINCT bus_id) FROM ml.trip_validity_trips;
    """)
    distinct_trip_bus_ids = cur.fetchone()[0]
    cur.execute("SELECT count(*) FROM ml.trip_validity_bus_avl_match;")
    match_rows = cur.fetchone()[0]
    print(
        "distinct trip bus_ids vs. match table rows:",
        distinct_trip_bus_ids,
        match_rows,
    )
    if distinct_trip_bus_ids != match_rows:
        msg = "bus_avl_match doesn't cover every distinct bus_id in trips"
        raise AssertionError(msg)

    # avl_match_source must be NULL exactly when avl_matched is false
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_bus_avl_match
        WHERE avl_matched <> (avl_match_source IS NOT NULL);
    """)
    inconsistent = cur.fetchone()[0]
    print("bus rows with avl_matched/avl_match_source disagreement:", inconsistent)
    if inconsistent != 0:
        msg = "avl_matched and avl_match_source disagree on some rows"
        raise AssertionError(msg)

    # positions must only exist for matched trips
    cur.execute("""
        SELECT count(*) FROM ml.trip_validity_trip_positions pos
        JOIN ml.trip_validity_trips t ON t.trip_id = pos.trip_id
        JOIN ml.trip_validity_bus_avl_match m ON m.bus_id = t.bus_id
        WHERE NOT m.avl_matched;
    """)
    orphan_positions = cur.fetchone()[0]
    print("position rows belonging to an unmatched trip:", orphan_positions)
    if orphan_positions != 0:
        msg = "found position rows for unmatched trips"
        raise AssertionError(msg)

    cur.execute("SELECT count(*) FROM ml.trip_validity_trip_positions;")
    total_positions = cur.fetchone()[0]
    cur.execute("""
        SELECT count(DISTINCT trip_id) FROM ml.trip_validity_trip_positions;
    """)
    trips_with_positions = cur.fetchone()[0]
    print(
        f"total positions: {total_positions}, "
        f"trips with >=1 position: {trips_with_positions}, "
        f"avg positions/trip: {total_positions / trips_with_positions:.1f}"
    )

    # spot-check: a real matched trip's positions stay within its own window
    cur.execute("""
        SELECT t.trip_id, t.trip_opening_timestamp, t.trip_closing_timestamp,
               min(pos.metric_timestamp), max(pos.metric_timestamp), count(*)
        FROM ml.trip_validity_trip_positions pos
        JOIN ml.trip_validity_trips t ON t.trip_id = pos.trip_id
        GROUP BY t.trip_id, t.trip_opening_timestamp, t.trip_closing_timestamp
        ORDER BY count(*) DESC
        LIMIT 1;
    """)
    trip_id, opened, closed, first_ping, last_ping, n = cur.fetchone()
    print(
        f"spot-check trip {trip_id}: window [{opened}, {closed}], "
        f"{n} pings spanning [{first_ping}, {last_ping}]"
    )
    if first_ping < opened or last_ping > closed:
        msg = "a trip's positions fall outside its own window"
        raise AssertionError(msg)

print("all checks passed")

distinct trip bus_ids vs. match table rows: 1730 1730
bus rows with avl_matched/avl_match_source disagreement: 0


position rows belonging to an unmatched trip: 0


total positions: 80567162, trips with >=1 position: 655401, avg positions/trip: 122.9


spot-check trip 557298: window [2023-11-16 12:04:42+00:00, 2023-11-21 16:57:58+00:00], 18866 pings spanning [2023-11-16 12:04:43+00:00, 2023-11-21 16:57:47+00:00]
all checks passed
